# Variant 3 — Advanced Architecture + **Data Augmentation + Ablation Study**

**Multi-task DistilBERT with CrossAttention + GraphReasoning + ClauseHierarchy**

This notebook does three things:
1. **Data Augmentation** — EDA on the Partially Evasive minority class (same as V1/V2)
2. **Full V3 training** — with augmented data, also adds stronger L2 regularisation and gradient accumulation
3. **Ablation Study** — trains 4 ablated variants of V3 to isolate which component causes overfitting:
   - **V3-A**: No Graph Reasoning (CrossAttn + ClausePool only)
   - **V3-B**: No Cross-Attention (Graph + ClausePool only)
   - **V3-C**: No Clause Hierarchy (Gating only = V2 equivalent within V3 framework)
   - **V3-D**: No Gating (CrossAttn + Graph + ClausePool, no token gate)

Results table is printed at the end for direct comparison.

In [1]:
!pip install transformers scikit-learn pandas torch tqdm --quiet
print("Libraries installed.")

Libraries installed.


In [2]:
import os, re, random, warnings, itertools, copy
import pandas as pd, numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type=="cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

DATA_DIR   = "/kaggle/input/datasets/pranavsai4205/no-gating"
OUTPUT_DIR = "/kaggle/working"

BERT_MODEL     = "distilbert-base-uncased"
MAX_LEN        = 256
STANCE_MAX_LEN = 128
Q_MAX_LEN      = 64
MAX_CLAUSES    = 6
CLAUSE_LEN     = 64
BATCH_SIZE     = 16
EPOCHS         = 5
DROPOUT        = 0.3
WARMUP_RATIO   = 0.1
FREEZE_EPOCHS  = 2
LR_EVASION     = 1e-5
LR_STANCE      = 2e-5
GRAD_ACCUM     = 2      # gradient accumulation steps (effective batch = 32)

EVASION_LABELS   = ["Non-Evasive", "Partially Evasive", "Evasive"]
EVASION_LABEL2ID = {l: i for i, l in enumerate(EVASION_LABELS)}
STANCE_LABELS    = ["FAVOR", "AGAINST"]
STANCE_LABEL2ID  = {l: i for i, l in enumerate(STANCE_LABELS)}

EVASION_CSV       = f"{DATA_DIR}/QAEvasion.csv"
STANCE_TRAIN_CSVS = [f"{DATA_DIR}/raw_train_biden.csv",f"{DATA_DIR}/raw_train_trump.csv",f"{DATA_DIR}/raw_train_bernie.csv"]
STANCE_VAL_CSVS   = [f"{DATA_DIR}/raw_val_biden.csv",  f"{DATA_DIR}/raw_val_trump.csv",  f"{DATA_DIR}/raw_val_bernie.csv"]
STANCE_TEST_CSVS  = [f"{DATA_DIR}/raw_test_biden.csv", f"{DATA_DIR}/raw_test_trump.csv", f"{DATA_DIR}/raw_test_bernie.csv"]

print("Configuration loaded.")
print(f"  Batch: {BATCH_SIZE}  GradAccum: {GRAD_ACCUM}  EffBatch: {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Epochs: {EPOCHS}  BERT frozen first {FREEZE_EPOCHS} epochs")

Device: cuda
GPU: Tesla T4
Configuration loaded.
  Batch: 16  GradAccum: 2  EffBatch: 32
  Epochs: 5  BERT frozen first 2 epochs


In [3]:
# ─────────────────────────────────────────────────────────────
# DATA AUGMENTATION MODULE
# ─────────────────────────────────────────────────────────────
SYNONYM_MAP = {
    "question": ["query", "inquiry"], "answer": ["response", "reply"],
    "policy": ["plan", "strategy"], "government": ["administration", "authority"],
    "issue": ["matter", "concern", "problem"], "important": ["crucial", "significant", "vital"],
    "believe": ["think", "consider", "feel"], "people": ["citizens", "individuals", "public"],
    "country": ["nation", "state"], "support": ["back", "endorse", "advocate"],
    "change": ["shift", "reform", "alter"], "need": ["require", "must"],
    "work": ["function", "operate", "effort"], "make": ["create", "produce", "form"],
    "good": ["beneficial", "positive", "effective"], "said": ["stated", "noted", "mentioned"],
    "know": ["understand", "recognize", "realize"], "think": ["believe", "consider", "feel"],
    "want": ["desire", "seek", "aim"], "time": ["period", "moment", "point"],
    "new": ["recent", "fresh", "updated"], "political": ["governmental", "civic"],
    "years": ["decades", "period"], "major": ["significant", "key", "primary"],
    "different": ["various", "distinct", "alternative"],
}

def eda_synonym_replace(text, n=3, seed=None):
    if seed is not None: random.seed(seed)
    words = text.split()
    replaceable = [(i, w.lower().strip('.,!?;:')) for i, w in enumerate(words)
                   if w.lower().strip('.,!?;:') in SYNONYM_MAP]
    if not replaceable: return text
    chosen = random.sample(replaceable, min(n, len(replaceable)))
    new_words = words[:]
    for i, w in chosen:
        syn = random.choice(SYNONYM_MAP[w])
        if words[i][0].isupper(): syn = syn.capitalize()
        new_words[i] = syn
    return " ".join(new_words)

def eda_random_swap(text, n=2, seed=None):
    if seed is not None: random.seed(seed)
    words = text.split()
    if len(words) < 4: return text
    new_words = words[:]
    for _ in range(n):
        i, j = random.sample(range(len(new_words)), 2)
        new_words[i], new_words[j] = new_words[j], new_words[i]
    return " ".join(new_words)

def eda_random_deletion(text, p=0.1, seed=None):
    if seed is not None: random.seed(seed)
    words = text.split()
    if len(words) <= 4: return text
    new_words = [w for w in words if random.random() > p]
    return " ".join(new_words) if new_words else text

def augment_pe_samples(questions, answers, labels, target_label_id=1, augment_factor=8, seed=42):
    aug_q, aug_a, aug_l = [], [], []
    pe_indices = [i for i, l in enumerate(labels) if l == target_label_id]
    print(f"  Augmenting {len(pe_indices)} PE samples (factor={augment_factor})...")

    ops = [
        lambda a, s: eda_synonym_replace(a, n=3, seed=s),
        lambda a, s: eda_random_swap(a, n=2, seed=s),
        lambda a, s: eda_random_deletion(a, p=0.1, seed=s),
        lambda a, s: eda_synonym_replace(eda_random_swap(a, n=1, seed=s), n=2, seed=s+1),
    ]

    for i, idx in enumerate(pe_indices):
        q, a, l = questions[idx], answers[idx], labels[idx]
        for k in range(augment_factor):
            op = ops[k % len(ops)]
            aug_q.append(q)
            aug_a.append(op(a, seed + i * 100 + k))   # <-- positional, not keyword
            aug_l.append(l)

    print(f"  Generated {len(aug_l)} augmented PE samples.")
    return aug_q, aug_a, aug_l


print("Augmentation module ready.")

Augmentation module ready.


In [4]:
def map_evasion_label(raw):
    raw = str(raw).strip()
    if raw.startswith("1."): return "Non-Evasive"
    elif raw == "2.3 Partial/half-answer": return "Partially Evasive"
    else: return "Evasive"

def load_evasion(filepath):
    print("Loading QA Evasion...")
    df = pd.read_csv(filepath)
    df["question"] = df["interview_question"].fillna("").str.strip()
    df["answer"]   = df["interview_answer"].fillna("").str.strip()
    df["coarse_label"] = df["label"].apply(map_evasion_label)
    df["label_id"]     = df["coarse_label"].map(EVASION_LABEL2ID).astype(int)
    df = df.dropna(subset=["question","answer","label_id"]).reset_index(drop=True)
    idx = list(range(len(df))); lbl = df["label_id"].tolist()
    itr,itmp,_,ytmp = train_test_split(idx,lbl,test_size=0.20,random_state=SEED,stratify=lbl)
    iva,ite,_,_     = train_test_split(itmp,ytmp,test_size=0.50,random_state=SEED,stratify=ytmp)
    q,a,l = df["question"].tolist(),df["answer"].tolist(),df["label_id"].tolist()
    def ex(ii): return [q[i] for i in ii],[a[i] for i in ii],[l[i] for i in ii]
    tr,va,te = ex(itr),ex(iva),ex(ite)
    print(f"  Train:{len(itr)}  Val:{len(iva)}  Test:{len(ite)}")
    return tr,va,te

def load_stance_split(filepaths, split_name):
    dfs=[]
    for fp in filepaths:
        if os.path.exists(fp):
            dfs.append(pd.read_csv(fp))
    df=pd.concat(dfs,ignore_index=True)
    df=df[df["Stance"].isin(STANCE_LABEL2ID)].reset_index(drop=True)
    df["label_id"]=df["Stance"].map(STANCE_LABEL2ID).astype(int)
    df["target"]=df["Target"].fillna("").str.strip()
    df["tweet"]=df["Tweet"].fillna("").str.strip()
    df=df.dropna(subset=["target","tweet","label_id"]).reset_index(drop=True)
    return df["target"].tolist(),df["tweet"].tolist(),df["label_id"].tolist()

print("="*60)
(Qetr,Aetr,yetr),(Qeva,Aeva,yeva),(Qete,Aete,yete)=load_evasion(EVASION_CSV)

print()
print("Applying EDA augmentation to training set (PE class)...")
aug_q, aug_a, aug_l = augment_pe_samples(Qetr, Aetr, yetr, target_label_id=1, augment_factor=8)
Qetr_aug = Qetr + aug_q; Aetr_aug = Aetr + aug_a; yetr_aug = yetr + aug_l
combined = list(zip(Qetr_aug, Aetr_aug, yetr_aug))
random.seed(SEED); random.shuffle(combined)
Qetr_aug, Aetr_aug, yetr_aug = [list(x) for x in zip(*combined)]
print(f"  Augmented evasion train: {len(yetr_aug)} samples")
for lid, lbl in enumerate(EVASION_LABELS):
    n = sum(1 for y in yetr_aug if y == lid)
    print(f"  {lbl:22s}: {n:4d} ({n/len(yetr_aug)*100:.1f}%)")

Qstr,Astr,ystr=load_stance_split(STANCE_TRAIN_CSVS,"train")
Qsva,Asva,ysva=load_stance_split(STANCE_VAL_CSVS,"val")
Qste,Aste,yste=load_stance_split(STANCE_TEST_CSVS,"test")
print("="*60)
print(f"Stance train: {len(ystr)}  val: {len(ysva)}  test: {len(yste)}")
print("Datasets ready.")

Loading QA Evasion...
  Train:2758  Val:345  Test:345

Applying EDA augmentation to training set (PE class)...
  Augmenting 63 PE samples (factor=8)...
  Generated 504 augmented PE samples.
  Augmented evasion train: 3262 samples
  Non-Evasive           : 1232 (37.8%)
  Partially Evasive     :  567 (17.4%)
  Evasive               : 1463 (44.8%)
Stance train: 17224  val: 2193  test: 2157
Datasets ready.


In [5]:
tokenizer = DistilBertTokenizer.from_pretrained(BERT_MODEL)

def split_into_clauses(text):
    parts = re.split(
        r'(?<=[.!?])\s+|(?:\s+(?:but|however|although|whereas|while|yet|'
        r'because|since|so|therefore|nevertheless|nonetheless)\s+)', text)
    parts = [p.strip() for p in parts if len(p.strip()) > 5]
    return parts[:MAX_CLAUSES] if parts else [text[:500]]

class TaskDataset(Dataset):
    def __init__(self, ta, tb, labels, max_len, task_id):
        self.ta,self.tb,self.labels,self.max_len,self.task_id = ta,tb,labels,max_len,task_id
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc = tokenizer(self.ta[idx],self.tb[idx],truncation=True,
                        padding="max_length",max_length=self.max_len,return_tensors="pt")
        qenc = tokenizer(self.ta[idx],truncation=True,
                         padding="max_length",max_length=Q_MAX_LEN,return_tensors="pt")
        clauses = split_into_clauses(self.tb[idx]) if self.task_id==0 else [self.tb[idx]]
        ids_l, masks_l = [], []
        for c in clauses[:MAX_CLAUSES]:
            ce = tokenizer(self.ta[idx],c,truncation=True,
                           padding="max_length",max_length=CLAUSE_LEN,return_tensors="pt")
            ids_l.append(ce["input_ids"].squeeze(0))
            masks_l.append(ce["attention_mask"].squeeze(0))
        n_real = len(ids_l)
        while len(ids_l) < MAX_CLAUSES:
            ids_l.append(torch.zeros(CLAUSE_LEN,dtype=torch.long))
            masks_l.append(torch.zeros(CLAUSE_LEN,dtype=torch.long))
        return {
            "input_ids":       enc["input_ids"].squeeze(0),
            "attention_mask":  enc["attention_mask"].squeeze(0),
            "q_input_ids":     qenc["input_ids"].squeeze(0),
            "q_attention_mask":qenc["attention_mask"].squeeze(0),
            "clause_ids":      torch.stack(ids_l),
            "clause_masks":    torch.stack(masks_l),
            "n_clauses":       torch.tensor(n_real, dtype=torch.long),
            "labels":          torch.tensor(self.labels[idx], dtype=torch.long),
            "task_id":         torch.tensor(self.task_id, dtype=torch.long),
        }

def make_loader(ta,tb,y,tid,ml,shuffle):
    return DataLoader(TaskDataset(ta,tb,y,ml,tid),batch_size=BATCH_SIZE,
                      shuffle=shuffle,num_workers=2,pin_memory=True)

loaders = {
    "evasion_train": make_loader(Qetr_aug,Aetr_aug,yetr_aug,0,MAX_LEN,True),
    "evasion_val":   make_loader(Qeva,Aeva,yeva,0,MAX_LEN,False),
    "evasion_test":  make_loader(Qete,Aete,yete,0,MAX_LEN,False),
    "stance_train":  make_loader(Qstr,Astr,ystr,1,STANCE_MAX_LEN,True),
    "stance_val":    make_loader(Qsva,Asva,ysva,1,STANCE_MAX_LEN,False),
    "stance_test":   make_loader(Qste,Aste,yste,1,STANCE_MAX_LEN,False),
}
for k,v in loaders.items():
    print(f"  {k:<20} {len(v.dataset):>6,} samples  {len(v):>5,} batches")
print("DataLoaders ready.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  evasion_train         3,262 samples    204 batches
  evasion_val             345 samples     22 batches
  evasion_test            345 samples     22 batches
  stance_train         17,224 samples  1,077 batches
  stance_val            2,193 samples    138 batches
  stance_test           2,157 samples    135 batches
DataLoaders ready.


In [6]:
# ─────────────────────────────────────────────────────────────
# MODEL ARCHITECTURE COMPONENTS
# ─────────────────────────────────────────────────────────────

def mean_pooling(h, mask):
    m = mask.unsqueeze(-1).expand(h.size()).float()
    return torch.sum(h*m,1) / torch.clamp(m.sum(1),min=1e-9)

class TokenLevelGating(nn.Module):
    def __init__(self, H, dr):
        super().__init__()
        self.Wa = nn.Linear(H, H, bias=False)
        self.Wq = nn.Linear(H, H, bias=True)
        self.drop = nn.Dropout(dr)
    def forward(self, h_seq, attention_mask, q_rep):
        gate = torch.sigmoid((self.Wa(h_seq) + self.Wq(q_rep).unsqueeze(1)) / 0.7)
        h_gated = self.drop(gate * h_seq)
        m = attention_mask.unsqueeze(-1).expand(h_gated.size()).float()
        return torch.sum(h_gated * m, 1) / torch.clamp(m.sum(1), min=1e-9)

class CrossAttention(nn.Module):
    def __init__(self, H):
        super().__init__()
        self.Wq = nn.Linear(H,H); self.Wk = nn.Linear(H,H); self.Wv = nn.Linear(H,H)
    def forward(self, q, clause_reps, n_clauses):
        B, C, H = clause_reps.size()
        Q = self.Wq(q).unsqueeze(1)
        K = self.Wk(clause_reps); V = self.Wv(clause_reps)
        scale = torch.sqrt(torch.tensor(H, dtype=torch.float32, device=q.device))
        scores = torch.bmm(Q, K.transpose(1,2)) / scale
        valid = torch.arange(C, device=q.device).unsqueeze(0) < n_clauses.unsqueeze(1)
        scores = scores.squeeze(1).masked_fill(~valid, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = weights / (weights.sum(-1, keepdim=True) + 1e-9)
        weights = F.dropout(weights, p=0.1, training=self.training)
        return torch.bmm(weights.unsqueeze(1), V).squeeze(1)

class GraphReasoningLayer(nn.Module):
    def __init__(self, H, dr, rounds=2):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(H*2,H), nn.ReLU(), nn.Dropout(dr))
            for _ in range(rounds)])
        self.norm = nn.LayerNorm(H)
    def forward(self, x, n_clauses):
        B, C, H = x.size()
        valid = (torch.arange(C, device=x.device).unsqueeze(0) < n_clauses.unsqueeze(1)).float()
        for layer in self.layers:
            sum_x = (x * valid.unsqueeze(-1)).sum(1, keepdim=True)
            denom = (n_clauses.unsqueeze(-1).unsqueeze(-1) - 1).clamp(min=1)
            nbr_mean = (sum_x - x * valid.unsqueeze(-1)) / denom
            x = self.norm(x + layer(torch.cat([x, nbr_mean], dim=-1)) * valid.unsqueeze(-1))
        return x

class ClauseAttentionPooling(nn.Module):
    def __init__(self, H, dr):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(H,H), nn.Tanh(), nn.Linear(H,1))
        self.norm  = nn.LayerNorm(H)
        self.drop  = nn.Dropout(dr)
    def forward(self, x, n_clauses):
        B, C, _ = x.size()
        valid = (torch.arange(C, device=x.device).unsqueeze(0) < n_clauses.unsqueeze(1))
        s = self.score(x).squeeze(-1).masked_fill(~valid, float("-inf"))
        w = F.softmax(s, dim=-1).unsqueeze(-1)
        return self.norm(self.drop((w * x).sum(1)))

class ClassHead(nn.Module):
    def __init__(self, H, n, dr):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout(dr), nn.Linear(H,256), nn.ReLU(),
                                 nn.Dropout(dr), nn.Linear(256,n))
    def forward(self, x): return self.net(x)

print("Architecture components defined.")
print("  TokenLevelGating | CrossAttention | GraphReasoningLayer | ClauseAttentionPooling | ClassHead")

Architecture components defined.
  TokenLevelGating | CrossAttention | GraphReasoningLayer | ClauseAttentionPooling | ClassHead


In [7]:
# ─────────────────────────────────────────────────────────────
# CONFIGURABLE V3 MODEL
# use_gating / use_cross_attn / use_graph / use_clause_pool
# can each be toggled for ablation
# ─────────────────────────────────────────────────────────────

class MultiTaskDistilBERT_V3(nn.Module):
    """
    Configurable V3 architecture for ablation study.

    Flags:
      use_gating      : TokenLevelGating on full sequence (from V2)
      use_cross_attn  : CrossAttention of q over clause representations
      use_graph       : GraphReasoningLayer message passing across clauses
      use_clause_pool : ClauseAttentionPooling weighted sum of clause reps

    When all four are True this is the original V3 (full).
    Toggling flags off produces the four ablated variants.

    Fallback: if no clause component is active (cross_attn=graph=clause_pool=False),
    the model degrades to gating-only (essentially V2).
    If gating is also off, it degrades to V1 mean-pool.
    """
    def __init__(self, ew, sw,
                 use_gating=True, use_cross_attn=True,
                 use_graph=True,  use_clause_pool=True):
        super().__init__()
        self.use_gating      = use_gating
        self.use_cross_attn  = use_cross_attn
        self.use_graph       = use_graph
        self.use_clause_pool = use_clause_pool

        self.encoder     = DistilBertModel.from_pretrained(BERT_MODEL)
        H = self.encoder.config.hidden_size
        self.clause_proj = nn.Linear(H, H)

        if use_cross_attn:  self.cross_attn = CrossAttention(H)
        if use_graph:       self.graph      = GraphReasoningLayer(H, DROPOUT, rounds=2)
        if use_clause_pool: self.cl_pool    = ClauseAttentionPooling(H, DROPOUT)
        if use_gating:
            self.evasion_gate = TokenLevelGating(H, DROPOUT)
            self.stance_gate  = TokenLevelGating(H, DROPOUT)

        # Count active streams for fusion weighting
        n_streams = (1 if use_gating else 1)  # always at least global mean pool
        if use_cross_attn:  n_streams += 1
        if use_clause_pool: n_streams += 1

        self.fusion = nn.LayerNorm(H)
        self.evasion_head    = ClassHead(H, len(EVASION_LABELS), DROPOUT)
        self.stance_head     = ClassHead(H, len(STANCE_LABELS),  DROPOUT)
        self.evasion_loss_fn = nn.CrossEntropyLoss(weight=ew)
        self.stance_loss_fn  = nn.CrossEntropyLoss(weight=sw)

    def _encode_clauses(self, clause_ids, clause_masks, B, C):
        ids   = clause_ids.view(B*C, -1)
        masks = clause_masks.view(B*C, -1)
        valid = masks.sum(dim=1) > 0
        if valid.sum() == 0:
            return torch.zeros(B, C, self.encoder.config.hidden_size, device=ids.device)
        valid_h = self.encoder(input_ids=ids[valid], attention_mask=masks[valid]).last_hidden_state
        pooled  = mean_pooling(valid_h, masks[valid])
        pool    = torch.zeros(ids.size(0), pooled.size(-1), device=ids.device)
        pool[valid] = pooled
        pool = F.relu(self.clause_proj(pool))
        return pool.view(B, C, -1)

    def freeze_inactive_head(self, task):
        task_modules = [
            [self.evasion_head] + ([self.evasion_gate] if self.use_gating else []),
            [self.stance_head]  + ([self.stance_gate]  if self.use_gating else []),
        ]
        shared = [self.encoder, self.clause_proj, self.fusion]
        if self.use_cross_attn:  shared.append(self.cross_attn)
        if self.use_graph:       shared.append(self.graph)
        if self.use_clause_pool: shared.append(self.cl_pool)

        for p in task_modules[1-task][0].parameters(): p.requires_grad=False
        if len(task_modules[1-task]) > 1:
            for p in task_modules[1-task][1].parameters(): p.requires_grad=False
        for p in task_modules[task][0].parameters(): p.requires_grad=True
        if len(task_modules[task]) > 1:
            for p in task_modules[task][1].parameters(): p.requires_grad=True
        for m in shared:
            for p in m.parameters(): p.requires_grad=True

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad=True

    def forward(self, input_ids, attention_mask, task_id, labels=None,
                q_input_ids=None, q_attention_mask=None,
                clause_ids=None, clause_masks=None, n_clauses=None):
        B    = input_ids.size(0)
        task = int(task_id[0].item())

        # Global encoding
        h     = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        q_h   = self.encoder(input_ids=q_input_ids, attention_mask=q_attention_mask).last_hidden_state
        q_rep = F.normalize(mean_pooling(q_h, q_attention_mask), dim=-1)

        # Base representation: gated or plain mean pool
        if self.use_gating:
            gate_fn = self.evasion_gate if task==0 else self.stance_gate
            base = gate_fn(h, attention_mask, q_rep)
        else:
            base = mean_pooling(h, attention_mask)

        # Clause pipeline
        C = clause_ids.size(1)
        clause_reps = self._encode_clauses(clause_ids, clause_masks, B, C)

        # Cross-attention stream
        xattn_out = self.cross_attn(q_rep, clause_reps, n_clauses) if self.use_cross_attn else None

        # Graph reasoning (updates clause_reps in-place for pooling)
        if self.use_graph:
            clause_reps = self.graph(clause_reps, n_clauses)

        # Clause pool stream
        cl_out = self.cl_pool(clause_reps, n_clauses) if self.use_clause_pool else None

        # Fusion: sum all active streams
        fused = base
        if xattn_out is not None: fused = fused + 0.7 * xattn_out
        if cl_out    is not None: fused = fused + 0.7 * cl_out
        fused = self.fusion(F.dropout(fused, p=0.2, training=self.training))

        head_fn = self.evasion_head if task==0 else self.stance_head
        logits  = head_fn(fused)
        loss_fn = self.evasion_loss_fn if task==0 else self.stance_loss_fn
        loss    = loss_fn(logits, labels) if labels is not None else None
        return loss, logits

# Pre-compute class weights
ew_base = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1,2]),y=yetr_aug),dtype=torch.float).to(device)
sw_base = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1]),  y=ystr),dtype=torch.float).to(device)
print(f"Evasion class weights (post-aug): {ew_base.tolist()}")
print(f"Stance  class weights: {sw_base.tolist()}")
print("\nConfigurable V3 model defined.")
print("Full V3:      use_gating=T  use_cross_attn=T  use_graph=T  use_clause_pool=T")
print("V3-A (NoGraph):   Gating + CrossAttn + ClausePool")
print("V3-B (NoCrossAttn): Gating + Graph + ClausePool")
print("V3-C (NoClause):    Gating only (= V2 within V3 framework)")
print("V3-D (NoGating):    CrossAttn + Graph + ClausePool  (no token gate)")

Evasion class weights (post-aug): [0.8825757503509521, 1.9176955223083496, 0.7432217001914978]
Stance  class weights: [1.0317479372024536, 0.9701475501060486]

Configurable V3 model defined.
Full V3:      use_gating=T  use_cross_attn=T  use_graph=T  use_clause_pool=T
V3-A (NoGraph):   Gating + CrossAttn + ClausePool
V3-B (NoCrossAttn): Gating + Graph + ClausePool
V3-C (NoClause):    Gating only (= V2 within V3 framework)
V3-D (NoGating):    CrossAttn + Graph + ClausePool  (no token gate)


In [8]:
# ─────────────────────────────────────────────────────────────
# GENERIC TRAINING + EVALUATION FUNCTIONS
# Reused for full V3 and all ablated variants
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate(model, loader, label_names):
    model.eval(); model.unfreeze_all()
    all_preds, all_labels, total_loss = [], [], 0.0
    for batch in loader:
        loss, logits = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
            batch["task_id"].to(device),
            batch["labels"].to(device),
            batch["q_input_ids"].to(device),
            batch["q_attention_mask"].to(device),
            batch["clause_ids"].to(device),
            batch["clause_masks"].to(device),
            batch["n_clauses"].to(device))
        total_loss += loss.item()
        all_preds.extend(torch.argmax(logits,1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
    return {
        "loss":   total_loss/len(loader),
        "acc":    accuracy_score(all_labels,all_preds),
        "f1":     f1_score(all_labels,all_preds,average="macro",zero_division=0),
        "prec":   precision_score(all_labels,all_preds,average="macro",zero_division=0),
        "rec":    recall_score(all_labels,all_preds,average="macro",zero_division=0),
        "report": classification_report(all_labels,all_preds,target_names=label_names,zero_division=0),
        "cm":     pd.DataFrame(confusion_matrix(all_labels,all_preds,labels=list(range(len(label_names)))),
                    index=[f"True:{l}" for l in label_names],
                    columns=[f"Pred:{l}" for l in label_names]),
        "preds": all_preds, "labels": all_labels,
    }

def print_eval(m, title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(f"  Loss:{m['loss']:.4f}  Acc:{m['acc']*100:.2f}%  MacroF1:{m['f1']:.4f}  Prec:{m['prec']:.4f}  Rec:{m['rec']:.4f}")
    print("\n  Per-class Report:")
    for line in m["report"].strip().split("\n"): print(f"  {line}")
    print("\n  Confusion Matrix:")
    print(m["cm"].to_string())
    print("="*60)

def train_model(model, variant_name, save_tag, n_epochs=EPOCHS):
    """Train a V3 variant and return best test metrics."""
    best_path = os.path.join(OUTPUT_DIR, f"best_{save_tag}.pt")

    # Build optimizer
    shared_params = list(model.encoder.parameters()) + list(model.clause_proj.parameters()) + list(model.fusion.parameters())
    if hasattr(model, 'cross_attn'):  shared_params += list(model.cross_attn.parameters())
    if hasattr(model, 'graph'):       shared_params += list(model.graph.parameters())
    if hasattr(model, 'cl_pool'):     shared_params += list(model.cl_pool.parameters())

    ev_task = list(model.evasion_head.parameters())
    if hasattr(model, 'evasion_gate'): ev_task += list(model.evasion_gate.parameters())
    st_task = list(model.stance_head.parameters())
    if hasattr(model, 'stance_gate'):  st_task += list(model.stance_gate.parameters())

    optimizer = AdamW([
        {"params": ev_task,         "lr": LR_EVASION},
        {"params": st_task,         "lr": LR_STANCE},
        {"params": shared_params,   "lr": LR_EVASION},
    ], weight_decay=0.01)

    n_steps  = len(loaders["stance_train"]) * 3 * n_epochs
    n_warmup = int(WARMUP_RATIO * n_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, n_warmup, n_steps)

    # Freeze encoder initially
    for p in model.encoder.parameters(): p.requires_grad = False

    best_avg_f1 = 0.0
    history = []

    print(f"\n{'='*60}")
    print(f"  TRAINING: {variant_name}")
    print(f"{'='*60}")
    print(f"  Params: {sum(p.numel() for p in model.parameters()):,}")

    for epoch in range(1, n_epochs + 1):
        if epoch == FREEZE_EPOCHS + 1:
            for p in model.encoder.parameters(): p.requires_grad = True
            print(f"\n  Epoch {epoch}: BERT UNFROZEN.")

        model.train()
        run_e_loss = run_s_loss = 0.0
        n_e = n_s = 0

        e_cycle  = itertools.cycle(list(loaders["evasion_train"]))
        combined = []
        for s_batch in loaders["stance_train"]:
            combined.append(next(e_cycle))
            combined.append(next(e_cycle))
            combined.append(s_batch)

        total = len(combined)
        print(f"\n  Epoch {epoch}/{n_epochs} — {total} steps")
        optimizer.zero_grad()

        for step, batch in enumerate(combined, 1):
            task = int(batch["task_id"][0].item())
            model.freeze_inactive_head(task)
            loss, logits = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["task_id"].to(device),
                batch["labels"].to(device),
                batch["q_input_ids"].to(device),
                batch["q_attention_mask"].to(device),
                batch["clause_ids"].to(device),
                batch["clause_masks"].to(device),
                batch["n_clauses"].to(device))

            (loss / GRAD_ACCUM).backward()
            if task == 0: run_e_loss += loss.item(); n_e += 1
            else:         run_s_loss += loss.item(); n_s += 1

            if step % GRAD_ACCUM == 0 or step == total:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            done = int(30 * step / total)
            print(f"\r  [{'='*done}{'-'*(30-done)}] {step}/{total}", end="", flush=True)

        print()
        e_val = evaluate(model, loaders["evasion_val"], EVASION_LABELS)
        s_val = evaluate(model, loaders["stance_val"],  STANCE_LABELS)
        avg_f1 = (e_val["f1"] + s_val["f1"]) / 2
        is_best = avg_f1 > best_avg_f1

        print(f"  E.F1:{e_val['f1']:.4f} S.F1:{s_val['f1']:.4f} AvgF1:{avg_f1:.4f}  E.ValLoss:{e_val['loss']:.4f}  {'<-BEST' if is_best else ''}")

        if is_best:
            best_avg_f1 = avg_f1
            torch.save(model.state_dict(), best_path)

        history.append({"epoch":epoch,"e_f1":e_val["f1"],"s_f1":s_val["f1"],
                         "avg_f1":avg_f1,"e_val_loss":e_val["loss"]})

    # Load best and test
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.unfreeze_all()
    e_test = evaluate(model, loaders["evasion_test"], EVASION_LABELS)
    s_test = evaluate(model, loaders["stance_test"],  STANCE_LABELS)

    print(f"\n  BEST TEST — E.F1:{e_test['f1']:.4f}  S.F1:{s_test['f1']:.4f}  Avg:{(e_test['f1']+s_test['f1'])/2:.4f}")
    return e_test, s_test, history

print("Training + evaluation functions ready.")

Training + evaluation functions ready.


In [9]:
# ─────────────────────────────────────────────────────────────
# SECTION 1: FULL V3 + DATA AUGMENTATION
# ─────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("  FULL V3 + Data Augmentation")
print("="*60)

model_v3_full = MultiTaskDistilBERT_V3(
    ew_base, sw_base,
    use_gating=True, use_cross_attn=True,
    use_graph=True,  use_clause_pool=True
).to(device)

e_full, s_full, hist_full = train_model(
    model_v3_full,
    "V3-Full (Gating+CrossAttn+Graph+ClausePool) + Aug",
    "v3_full_aug"
)
print_eval(e_full, "EVASION TEST — V3 Full + Aug")
print_eval(s_full, "STANCE  TEST — V3 Full + Aug")


  FULL V3 + Data Augmentation


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  TRAINING: V3-Full (Gating+CrossAttn+Graph+ClausePool) + Aug
  Params: 74,437,894

  Epoch 1/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4108 S.F1:0.6753 AvgF1:0.5430  E.ValLoss:1.0668  <-BEST

  Epoch 2/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4029 S.F1:0.7407 AvgF1:0.5718  E.ValLoss:1.6950  <-BEST

  Epoch 3: BERT UNFROZEN.

  Epoch 3/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4007 S.F1:0.7373 AvgF1:0.5690  E.ValLoss:2.4077  

  Epoch 4/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4044 S.F1:0.7477 AvgF1:0.5761  E.ValLoss:3.0549  <-BEST

  Epoch 5/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4034 S.F1:0.7530 AvgF1:0.5782  E.ValLoss:2.6363  <-BEST

  BEST TEST — E.F1:0.4455  S.F1:0.7553  Avg:0.6004

  EVASION TEST — V3 Full + Aug
  Loss:2.3030  Acc:58.84%  MacroF1:0.4455  Prec:0.4617  Rec:0.4470

  Per-class Report:
  precision    recall  f1-score   support
 

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 2: ABLATION STUDY
# Each run removes ONE component. Compare vs full V3.
# ─────────────────────────────────────────────────────────────

ablation_configs = [
    {
        "name":   "V3-A: No Graph Reasoning",
        "tag":    "v3_ablation_no_graph",
        "flags":  dict(use_gating=True, use_cross_attn=True, use_graph=False, use_clause_pool=True),
        "desc":   "Removes GraphReasoningLayer. Keeps: Gating + CrossAttn + ClausePool.",
    },
    {
        "name":   "V3-B: No Cross-Attention",
        "tag":    "v3_ablation_no_crossattn",
        "flags":  dict(use_gating=True, use_cross_attn=False, use_graph=True, use_clause_pool=True),
        "desc":   "Removes CrossAttention. Keeps: Gating + Graph + ClausePool.",
    },
    {
        "name":   "V3-C: No Clause Hierarchy (Gating only)",
        "tag":    "v3_ablation_no_clause",
        "flags":  dict(use_gating=True, use_cross_attn=False, use_graph=False, use_clause_pool=False),
        "desc":   "Removes all clause components. Keeps: Gating only (= V2 baseline inside V3 framework).",
    },
    {
        "name":   "V3-D: No Gating",
        "tag":    "v3_ablation_no_gating",
        "flags":  dict(use_gating=False, use_cross_attn=True, use_graph=True, use_clause_pool=True),
        "desc":   "Removes TokenLevelGating. Keeps: CrossAttn + Graph + ClausePool.",
    },
]

ablation_results = {}

for cfg in ablation_configs:
    print(f"\n{'─'*60}")
    print(f"  ABLATION: {cfg['name']}")
    print(f"  {cfg['desc']}")
    print(f"{'─'*60}")

    model_abl = MultiTaskDistilBERT_V3(ew_base, sw_base, **cfg["flags"]).to(device)
    e_abl, s_abl, hist_abl = train_model(model_abl, cfg["name"], cfg["tag"])

    ablation_results[cfg["name"]] = {
        "e_f1":  e_abl["f1"],
        "s_f1":  s_abl["f1"],
        "avg":   (e_abl["f1"] + s_abl["f1"]) / 2,
        "acc":   e_abl["acc"],
        "report": e_abl["report"],
        "history": hist_abl,
    }

    print_eval(e_abl, f"EVASION TEST — {cfg['name']}")

print("\nAll ablation experiments complete.")


────────────────────────────────────────────────────────────
  ABLATION: V3-A: No Graph Reasoning
  Removes GraphReasoningLayer. Keeps: Gating + CrossAttn + ClausePool.
────────────────────────────────────────────────────────────


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  TRAINING: V3-A: No Graph Reasoning
  Params: 72,075,526

  Epoch 1/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.3990 S.F1:0.7066 AvgF1:0.5528  E.ValLoss:1.0508  <-BEST

  Epoch 2/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4199 S.F1:0.7388 AvgF1:0.5794  E.ValLoss:1.8355  <-BEST

  Epoch 3: BERT UNFROZEN.

  Epoch 3/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4246 S.F1:0.7439 AvgF1:0.5842  E.ValLoss:2.2105  <-BEST

  Epoch 4/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4145 S.F1:0.7532 AvgF1:0.5838  E.ValLoss:2.9578  

  Epoch 5/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4134 S.F1:0.7433 AvgF1:0.5784  E.ValLoss:3.1875  

  BEST TEST — E.F1:0.4462  S.F1:0.7694  Avg:0.6078

  EVASION TEST — V3-A: No Graph Reasoning
  Loss:1.9780  Acc:60.29%  MacroF1:0.4462  Prec:0.4524  Rec:0.4448

  Per-class Report:
  precision    recall  f1-score   support
  
        Non-Evasiv

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  TRAINING: V3-B: No Cross-Attention
  Params: 72,666,118

  Epoch 1/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.3635 S.F1:0.7099 AvgF1:0.5367  E.ValLoss:1.0653  <-BEST

  Epoch 2/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.3986 S.F1:0.7514 AvgF1:0.5750  E.ValLoss:1.7068  <-BEST

  Epoch 3: BERT UNFROZEN.

  Epoch 3/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4130 S.F1:0.7514 AvgF1:0.5822  E.ValLoss:2.5594  <-BEST

  Epoch 4/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4062 S.F1:0.7404 AvgF1:0.5733  E.ValLoss:2.5087  

  Epoch 5/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4134 S.F1:0.7492 AvgF1:0.5813  E.ValLoss:3.2066  

  BEST TEST — E.F1:0.4410  S.F1:0.7610  Avg:0.6010

  EVASION TEST — V3-B: No Cross-Attention
  Loss:2.2244  Acc:59.42%  MacroF1:0.4410  Prec:0.4418  Rec:0.4410

  Per-class Report:
  precision    recall  f1-score   support
  
        Non-Evasiv

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  TRAINING: V3-C: No Clause Hierarchy (Gating only)
  Params: 69,710,853

  Epoch 1/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4461 S.F1:0.7022 AvgF1:0.5742  E.ValLoss:0.9988  <-BEST

  Epoch 2/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4151 S.F1:0.7355 AvgF1:0.5753  E.ValLoss:1.4449  <-BEST

  Epoch 3: BERT UNFROZEN.

  Epoch 3/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4311 S.F1:0.7611 AvgF1:0.5961  E.ValLoss:1.5143  <-BEST

  Epoch 4/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4425 S.F1:0.7621 AvgF1:0.6023  E.ValLoss:2.1633  <-BEST

  Epoch 5/5 — 3231 steps
  [==================------------] 1947/3231

In [9]:
# ─────────────────────────────────────────────────────────────
# SECTION 2: ABLATION STUDY
# Each run removes ONE component. Compare vs full V3.
# ─────────────────────────────────────────────────────────────

ablation_configs = [
    {
        "name":   "V3-C: No Clause Hierarchy (Gating only)",
        "tag":    "v3_ablation_no_clause",
        "flags":  dict(use_gating=True, use_cross_attn=False, use_graph=False, use_clause_pool=False),
        "desc":   "Removes all clause components. Keeps: Gating only (= V2 baseline inside V3 framework).",
    },
    {
        "name":   "V3-D: No Gating",
        "tag":    "v3_ablation_no_gating",
        "flags":  dict(use_gating=False, use_cross_attn=True, use_graph=True, use_clause_pool=True),
        "desc":   "Removes TokenLevelGating. Keeps: CrossAttn + Graph + ClausePool.",
    },
]

ablation_results = {}

for cfg in ablation_configs:
    print(f"\n{'─'*60}")
    print(f"  ABLATION: {cfg['name']}")
    print(f"  {cfg['desc']}")
    print(f"{'─'*60}")

    model_abl = MultiTaskDistilBERT_V3(ew_base, sw_base, **cfg["flags"]).to(device)
    e_abl, s_abl, hist_abl = train_model(model_abl, cfg["name"], cfg["tag"])

    ablation_results[cfg["name"]] = {
        "e_f1":  e_abl["f1"],
        "s_f1":  s_abl["f1"],
        "avg":   (e_abl["f1"] + s_abl["f1"]) / 2,
        "acc":   e_abl["acc"],
        "report": e_abl["report"],
        "history": hist_abl,
    }

    print_eval(e_abl, f"EVASION TEST — {cfg['name']}")

print("\nAll ablation experiments complete.")


────────────────────────────────────────────────────────────
  ABLATION: V3-C: No Clause Hierarchy (Gating only)
  Removes all clause components. Keeps: Gating only (= V2 baseline inside V3 framework).
────────────────────────────────────────────────────────────


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  TRAINING: V3-C: No Clause Hierarchy (Gating only)
  Params: 69,710,853

  Epoch 1/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4253 S.F1:0.6841 AvgF1:0.5547  E.ValLoss:0.9710  <-BEST

  Epoch 2/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4359 S.F1:0.7358 AvgF1:0.5859  E.ValLoss:1.5407  <-BEST

  Epoch 3: BERT UNFROZEN.

  Epoch 3/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4303 S.F1:0.7545 AvgF1:0.5924  E.ValLoss:1.6006  <-BEST

  Epoch 4/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4515 S.F1:0.7573 AvgF1:0.6044  E.ValLoss:2.1253  <-BEST

  Epoch 5/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4121 S.F1:0.7571 AvgF1:0.5846  E.ValLoss:2.3325  

  BEST TEST — E.F1:0.5055  S.F1:0.7669  Avg:0.6362

  EVASION TEST — V3-C: No Clause Hierarchy (Gating only)
  Loss:1.9237  Acc:62.32%  MacroF1:0.5055  Prec:0.5128  Rec:0.5005

  Per-class Report:
  precision    recall  f1-s

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  TRAINING: V3-D: No Gating
  Params: 72,077,062

  Epoch 1/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.3720 S.F1:0.7054 AvgF1:0.5387  E.ValLoss:1.0606  <-BEST

  Epoch 2/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4127 S.F1:0.7465 AvgF1:0.5796  E.ValLoss:1.7183  <-BEST

  Epoch 3: BERT UNFROZEN.

  Epoch 3/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4105 S.F1:0.7550 AvgF1:0.5827  E.ValLoss:2.2430  <-BEST

  Epoch 4/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4083 S.F1:0.7574 AvgF1:0.5829  E.ValLoss:2.6942  <-BEST

  Epoch 5/5 — 3231 steps
  [==============================] 3231/3231
  E.F1:0.4025 S.F1:0.7483 AvgF1:0.5754  E.ValLoss:3.0316  

  BEST TEST — E.F1:0.4495  S.F1:0.7524  Avg:0.6010

  EVASION TEST — V3-D: No Gating
  Loss:2.3703  Acc:60.00%  MacroF1:0.4495  Prec:0.4520  Rec:0.4474

  Per-class Report:
  precision    recall  f1-score   support
  
        Non-Evasive       0.56

In [ ]:
# ─────────────────────────────────────────────────────────────
# SECTION 3: ABLATION RESULTS TABLE
# ─────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("  ABLATION STUDY — FINAL RESULTS TABLE")
print("="*70)

rows = [
    ("V3 Full + Aug",        e_full["f1"],   s_full["f1"],   (e_full["f1"]+s_full["f1"])/2,   e_full["acc"]),
]
for cfg in ablation_configs:
    r = ablation_results[cfg["name"]]
    rows.append((cfg["name"], r["e_f1"], r["s_f1"], r["avg"], r["acc"]))

print(f"  {'Variant':<40} {'Eva.F1':>8} {'Sta.F1':>8} {'Avg.F1':>8} {'Eva.Acc':>9}")
print(f"  {'-'*77}")
for row in rows:
    mark = " <-- BEST" if row[3] == max(r[3] for r in rows) else ""
    print(f"  {row[0]:<40} {row[1]:>8.4f} {row[2]:>8.4f} {row[3]:>8.4f} {row[4]*100:>8.2f}%{mark}")

print(f"  {'='*77}")
print()
print("  KEY FINDING:")
best_row = max(rows, key=lambda r: r[3])
print(f"    Best ablation variant: {best_row[0]}")
print(f"    => Avg Macro F1: {best_row[3]:.4f}")
print()

# Per-class F1 for Partially Evasive (the critical minority class)
print("  PARTIALLY EVASIVE F1 BREAKDOWN (minority class — most critical):")
print(f"  {'Variant':<40} {'PE F1':>8}")
print(f"  {'-'*52}")

def get_pe_f1(report_str):
    for line in report_str.split("\n"):
        if "Partially Evasive" in line:
            parts = line.split()
            try: return float(parts[3])  # F1 column
            except: return 0.0
    return 0.0

pe_full = get_pe_f1(e_full["report"])
print(f"  {'V3 Full + Aug':<40} {pe_full:>8.4f}")
for cfg in ablation_configs:
    r = ablation_results[cfg["name"]]
    pe = get_pe_f1(r["report"])
    print(f"  {cfg['name']:<40} {pe:>8.4f}")

print("="*70)

In [ ]:
# Save full results CSV
all_rows = [{"variant": rows[i][0], "eva_f1": rows[i][1], "sta_f1": rows[i][2],
             "avg_f1": rows[i][3], "eva_acc": rows[i][4]} for i in range(len(rows))]
pd.DataFrame(all_rows).to_csv(os.path.join(OUTPUT_DIR, "v3_ablation_results.csv"), index=False)

# Save full V3 test predictions
pd.DataFrame({"question":Qete,"answer":Aete,
    "true":[EVASION_LABELS[l] for l in e_full["labels"]],
    "pred":[EVASION_LABELS[p] for p in e_full["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,"results_v3_full_aug_evasion.csv"),index=False)

pd.DataFrame({"target":Qste,"tweet":Aste,
    "true":[STANCE_LABELS[l] for l in s_full["labels"]],
    "pred":[STANCE_LABELS[p] for p in s_full["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,"results_v3_full_aug_stance.csv"),index=False)

print("All results saved.")
print(f"  -> v3_ablation_results.csv")
print(f"  -> results_v3_full_aug_evasion.csv")
print(f"  -> results_v3_full_aug_stance.csv")